# Backup and Recovery
## Scenario 2: Without Writer Modifications or Maintenance Window
### Use case: Delta Lake Source Table with Full Migration

#### Configure Spark session with proper Delta Lake and Iceberg settings

In [ ]:
%%configure -f
{   
    "conf":{
        "numExecutors": 1, 
        "executorMemory": "14G",
        "executorCores": 4, 
        "spark.dynamicAllocation.enabled": "false",
        "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension",
        "spark.sql.catalog.iceberg_catalog": "org.apache.iceberg.spark.SparkCatalog",
        "spark.sql.catalog.iceberg_catalog.type": "glue",
        "spark.hadoop.hive.metastore.client.factory.class": "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory",
        "spark.jars": "/usr/share/aws/delta/lib/delta-spark.jar,/usr/share/aws/delta/lib/delta-storage.jar",
        "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    }
}

#### Configure variables

In [ ]:
delta_table_name = "cdc_delta_table"
iceberg_table_name="cdc_delta_table_iceberg"
bucket_name="mybucket"
db_name="mydb"

#### Create Delta table and populate

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{delta_table_name} (
  id INT,
  name STRING
)
USING DELTA
LOCATION 's3://{bucket_name}/{db_name}/{delta_table_name}/'
""")

#Enable ChangeDataFeed
spark.sql(f"""
ALTER TABLE {db_name}.{delta_table_name}
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');
""")

spark.sql(f"""
INSERT INTO {db_name}.{delta_table_name}
VALUES 
  (1, 'Alice'),
  (2, 'Bob')
""")

spark.sql(f"""
INSERT INTO {db_name}.{delta_table_name}
VALUES 
  (3, 'Stefano'),
  (4, 'Elio')
""")

#### Initial Migration Phase 

##### 1. Get latest Delta table commit timestamp

In [ ]:
# Get the latest commit timestamp from the Delta table
latest_commit_timestamp = spark.sql(f"DESCRIBE HISTORY {db_name}.{delta_table_name}") \
                         .select("timestamp") \
                         .orderBy("version", ascending=False) \
                         .first()[0]

##### 2. Perform full migration using time travel query as source data reading the state of the Delta table at `latest_commit_timestamp`

In [ ]:
spark.sql(f"""
CREATE TABLE iceberg_catalog.{db_name}.{iceberg_table_name}
USING ICEBERG
LOCATION 's3://{bucket_name}/{db_name}/{iceberg_table_name}/'
AS (SELECT * FROM {db_name}.{delta_table_name} TIMESTAMP AS OF '{latest_commit_timestamp}')
""")

In [ ]:
spark.sql(f"""
SELECT * FROM iceberg_catalog.{db_name}.{iceberg_table_name}
""").show()

##### 3. Checkpoint latest_commit_timestamp as latest_sync_timestamp (eg., in S3)

In [ ]:
import json, boto3, datetime
latest_commit_timestamp_str = latest_commit_timestamp.strftime('%Y%m%d%H%M%S')
s3 = boto3.client('s3')
s3.put_object(
    Body=json.dumps({"latest_sync_timestamp": latest_commit_timestamp_str}),
    Bucket=bucket_name,
    Key=f'{db_name}/{iceberg_table_name}/latest_sync.json',
    ContentType='application/json'
)

At this point we are done with the intial migration. Let’say some operations are performed in the source Delta table after the initial sync (eg., three MERGE operations):

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True)
])

dataframes = [
    spark.createDataFrame([
        (1, "Alice2"), # Update record
        (2, "Bob2"), # Update record
        (5, "John") # New record
    ], schema),
    spark.createDataFrame([
        (1, None),  # Delete record
        (6, "Michael")  # New record
    ], schema),
    spark.createDataFrame([
        (2, "Bob3"),     # Update record
        (5, None),       # Delete record
        (7, "Jennifer")  # New record
    ], schema)
]

for i, df in enumerate(dataframes, 1):
    df.createOrReplaceTempView(f"ingestion_{i}")
    
    # Execute the merge for the current DataFrame
    spark.sql(f"""
    MERGE INTO {db_name}.{delta_table_name} h
    USING ingestion_{i} s
    ON h.id = s.id
    WHEN MATCHED AND s.name IS NOT NULL THEN UPDATE SET * 
    WHEN MATCHED AND s.name IS NULL THEN DELETE
    WHEN NOT MATCHED THEN INSERT *
    """)
    spark.catalog.dropTempView(f"ingestion_{i}")

At this point this is the status of the Delta and Iceberg tables:

In [ ]:
spark.sql(f"""
SELECT id, name FROM {db_name}.{delta_table_name} ORDER BY id
""").show()

In [ ]:
spark.sql(f"""
SELECT * FROM iceberg_catalog.{db_name}.{iceberg_table_name} ORDER BY id
""").show()

#### Incremental Sync

##### 1. Retrieve latest sync time (`latest_sync_timestamp`) from the checkpoint location

In [ ]:
import json, boto3

s3 = boto3.client('s3')
response = s3.get_object(Bucket=bucket_name, Key=f'{db_name}/{iceberg_table_name}/latest_sync.json')
latest_sync_timestamp_str = json.loads(response['Body'].read())['latest_sync_timestamp']
latest_sync_timestamp = datetime.datetime.strptime(latest_sync_timestamp_str, '%Y%m%d%H%M%S')
latest_sync_timestamp

##### 2. Get the latest Delta table commit timestamp (`latest_commit_timestamp`)

In [ ]:
# Get the latest commit timestamp from the Delta table
latest_commit_timestamp = spark.sql(f"DESCRIBE HISTORY {db_name}.{delta_table_name}") \
                         .select("timestamp") \
                         .orderBy("version", ascending=False) \
                         .first()[0]
latest_commit_timestamp

##### 3. Get changed data in the Delta table between `latest_sync_timestamp` and `latest_commit_timestamp`

In [ ]:
cdc_df = spark.sql(f"""
SELECT *
FROM table_changes('{db_name}.{delta_table_name}', '{latest_sync_timestamp}', '{latest_commit_timestamp}')
WHERE _commit_timestamp > '{latest_sync_timestamp}'
""")

In [ ]:
cdc_df.show(truncate=False)

##### 4. Deduplicate CDC entries to get the latest updates/inserts/deletes

In [ ]:
from pyspark.sql.functions import col, when, lit, row_number
from pyspark.sql.window import Window

# Step 1: Filter out update_preimage records as they represent the previous state
filtered_cdc = cdc_df.filter(col("_change_type") != "update_preimage")

# Step 2: Create a window spec partitioned by id and ordered by commit version (descending)
window_spec = Window.partitionBy("id").orderBy(col("_commit_version").desc())

# Add row number to identify the most recent change for each id
ranked_cdc = filtered_cdc.withColumn("row_num", row_number().over(window_spec))

# Filter to keep only the most recent change for each id
latest_changes = ranked_cdc.filter(col("row_num") == 1)

# Step 3: Transform the data for the merge operation
merge_df = latest_changes.withColumn(
    "final_name",
    when(col("_change_type") == "delete", lit(None))
    .when(col("_change_type").isin("insert", "update_postimage"), col("name"))
    .otherwise(None)
).select(
    col("id"),
    col("final_name").alias("name")
)

# Register as temp view for the merge operation
merge_df.createOrReplaceTempView("iceberg_sync_data")

In [ ]:
merge_df.show()

##### 5. MERGE properly fomatted CDC data into the Iceberg table

In [ ]:
spark.sql(f"""
MERGE INTO iceberg_catalog.{db_name}.{iceberg_table_name} i
USING iceberg_sync_data s
ON i.id = s.id
WHEN MATCHED AND s.name IS NOT NULL THEN UPDATE SET * 
WHEN MATCHED AND s.name IS NULL THEN DELETE
WHEN NOT MATCHED AND s.name IS NOT NULL THEN INSERT *
""")

You can confirm by querying the Iceberg table it is not identical compared to the Delta Lake table.

In [ ]:
spark.sql(f"""
SELECT id, name FROM {db_name}.{delta_table_name} ORDER BY id
""").show()

In [ ]:
spark.sql(f"""
SELECT * FROM iceberg_catalog.{db_name}.{iceberg_table_name} ORDER BY id
""").show()

##### 6. Checkpoint `latest_commit_timestamp` as `last_sync_timestamp` (eg., in S3) for the next sync

In [ ]:
latest_commit_timestamp_str = latest_commit_timestamp.strftime('%Y%m%d%H%M%S')
s3 = boto3.client('s3')
s3.put_object(
    Body=json.dumps({"latest_sync_timestamp": latest_commit_timestamp_str}),
    Bucket=bucket_name,
    Key=f'{db_name}/{iceberg_table_name}/latest_sync.json',
    ContentType='application/json'
)